# Task 4 — Best model training + sampling + evaluation (Colab)

This notebook trains the **chosen best model** longer (default: **SP XL from Part 2 settings**), then:

- Generates **≥10 unconditional** samples (from an `<svg` prefix)
- Generates **≥5 prefix-conditioned** completions
- Uses **temperature** + **top-k/top-p** sampling
- Computes quantitative metrics: **test perplexity**, **XML validity**, **structural validity**, **render rate**
- Builds a **rendered grid** figure for the report

By default, Task 4 outputs (checkpoints, samples, eval JSON, report figures) go under
`/content/drive/MyDrive/svg_task4_outputs/...` so a disconnect is less likely to delete them. The
repo clone under `REPO_ROOT` is used to run the Python modules.


## 1) Clone repo & set `REPO_ROOT`


In [2]:
import os, subprocess, sys

CLONE_URL = os.environ.get("TASK_REPO_URL", "https://github.com/yuelihe2-svg/svg-scaling-laws-transformers.git")

os.chdir("/content")
subprocess.run(["rm", "-rf", "svg-scaling-laws-transformers"], check=False)
subprocess.check_call(["git", "clone", CLONE_URL])

REPO_ROOT = "/content/svg-scaling-laws-transformers"
assert os.path.exists(os.path.join(REPO_ROOT, "scripts", "task4", "train_best_model.py"))
os.environ["REPO_ROOT"] = REPO_ROOT
os.chdir(REPO_ROOT)
print("REPO_ROOT:", REPO_ROOT)
print("cwd:", os.getcwd())


REPO_ROOT: /content/svg-scaling-laws-transformers
cwd: /content/svg-scaling-laws-transformers


## 2) Install deps


In [3]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])
import torch
print("torch", torch.__version__, "cuda", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))


torch 2.10.0+cu128 cuda True
NVIDIA H100 80GB HBM3


## 3) Mount Drive and set Part 1 data paths


In [4]:
from google.colab import drive
drive.mount("/content/drive")

import os

# ===== data on Drive =====
DATA_ROOT = "/content/drive/MyDrive/svg_task2_data"
TRAIN = os.path.join(DATA_ROOT, "train.jsonl")
VAL   = os.path.join(DATA_ROOT, "val.jsonl")
TEST  = os.path.join(DATA_ROOT, "test.jsonl")
SPM   = os.path.join(DATA_ROOT, "spm.model")

# ===== Task 4: train locally first, copy to Drive only at the end =====
TASK4_LOCAL = "/content/task4_outputs_e7"
TASK4_DRIVE = "/content/drive/MyDrive/svg_task4_outputs_e7"

MODEL_DIR   = os.path.join(TASK4_LOCAL, "best_xl_sp")
SAMPLES_DIR = os.path.join(TASK4_LOCAL, "samples")
FIG_DIR     = os.path.join(TASK4_LOCAL, "figures_report")
EVAL_JSON   = os.path.join(TASK4_LOCAL, "eval_metrics.json")

for p in (TRAIN, VAL, SPM):
    print(p, os.path.exists(p))
assert os.path.exists(TRAIN) and os.path.exists(VAL) and os.path.exists(SPM)

print("TASK4_LOCAL =", TASK4_LOCAL)
print("TASK4_DRIVE =", TASK4_DRIVE)
print("MODEL_DIR   =", MODEL_DIR)

Mounted at /content/drive
/content/drive/MyDrive/svg_task2_data/train.jsonl True
/content/drive/MyDrive/svg_task2_data/val.jsonl True
/content/drive/MyDrive/svg_task2_data/spm.model True
TASK4_LOCAL = /content/task4_outputs_e7
TASK4_DRIVE = /content/drive/MyDrive/svg_task4_outputs_e7
MODEL_DIR   = /content/task4_outputs_e7/best_xl_sp


## 4) Train best model longer (SP XL) + checkpoints


In [6]:
import os, sys, shutil, subprocess

REPO_ROOT = os.environ["REPO_ROOT"]

# ===== fresh run: remove old local outputs =====
OUT_DIR = MODEL_DIR
if os.path.exists(OUT_DIR):
    shutil.rmtree(OUT_DIR)
os.makedirs(OUT_DIR, exist_ok=True)

cmd = [
    sys.executable, "-u", "-m", "scripts.task4.train_best_model",
    "--train-jsonl", TRAIN,
    "--val-jsonl", VAL,
    "--spm-model", SPM,
    "--preset", "xl",
    "--lr", "0.003",
    "--tokens-per-batch", "32768",
    "--block-size", "512",
    "--epochs", "7",
    "--save-every-steps", "0",
    "--out-dir", OUT_DIR,
]

print("Starting fresh training")
print("Running:", " ".join(cmd))

proc = subprocess.Popen(
    cmd,
    cwd=REPO_ROOT,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

for line in proc.stdout:
    print(line, end="")

ret = proc.wait()
if ret != 0:
    raise RuntimeError(f"Training failed with exit code {ret}")

print("Done. OUT_DIR =", OUT_DIR)

Starting fresh training
Running: /usr/bin/python3 -u -m scripts.task4.train_best_model --train-jsonl /content/drive/MyDrive/svg_task2_data/train.jsonl --val-jsonl /content/drive/MyDrive/svg_task2_data/val.jsonl --spm-model /content/drive/MyDrive/svg_task2_data/spm.model --preset xl --lr 0.003 --tokens-per-batch 32768 --block-size 512 --epochs 7 --save-every-steps 0 --out-dir /content/task4_outputs_e7/best_xl_sp
Loading train stream ...
Loading val stream ...
Train tokens (incl. EOS): 103,255,934 | Val: 1,052,909
Preset=xl ~~88M | trainable params: 90,842,112
batch_size=64 block_size=512 tokens/step=32768
steps_per_epoch=3151 epochs=7 -> max_steps=22057
step 0/22057 train_loss=8.2546 lr=1.50e-06 tok/s=56769
step 50/22057 train_loss=3.7100 lr=7.65e-05 tok/s=2530402
step 100/22057 train_loss=3.0514 lr=1.51e-04 tok/s=2756781
step 150/22057 train_loss=2.7655 lr=2.27e-04 tok/s=2520560
step 200/22057 train_loss=2.4568 lr=3.02e-04 tok/s=2523942
step 250/22057 train_loss=2.2332 lr=3.76e-04 tok/

## 5) Generate samples (SVG + PNG)


In [16]:
import os, sys, subprocess, shutil

REPO_ROOT = os.environ["REPO_ROOT"]

# remove old samples
if os.path.exists(SAMPLES_DIR):
    shutil.rmtree(SAMPLES_DIR)
os.makedirs(SAMPLES_DIR, exist_ok=True)

CKPT = os.path.join(MODEL_DIR, "checkpoints", "final.pt")
CFG  = os.path.join(MODEL_DIR, "config.json")

print("Using MODEL_DIR =", MODEL_DIR)
print("Using CKPT =", CKPT)
print("Using CFG =", CFG)

cmd = [
    sys.executable, "-u", "-m", "scripts.task4.sample_generate",
    "--spm-model", SPM,
    "--checkpoint", CKPT,
    "--config", CFG,
    "--out-dir", SAMPLES_DIR,
    "--num-uncond", "10",
    "--num-prefix", "5",
    "--temperatures", "0.5", "0.8", "1.0",
    "--top-k", "50",
    "--top-p", "0.95",
    "--render",
]

print("Running:", " ".join(cmd))
subprocess.check_call(cmd, cwd=REPO_ROOT)

print("Done. SAMPLES_DIR =", SAMPLES_DIR)

Using MODEL_DIR = /content/task4_outputs_e7/best_xl_sp
Using CKPT = /content/task4_outputs_e7/best_xl_sp/checkpoints/final.pt
Using CFG = /content/task4_outputs_e7/best_xl_sp/config.json
Running: /usr/bin/python3 -u -m scripts.task4.sample_generate --spm-model /content/drive/MyDrive/svg_task2_data/spm.model --checkpoint /content/task4_outputs_e7/best_xl_sp/checkpoints/final.pt --config /content/task4_outputs_e7/best_xl_sp/config.json --out-dir /content/task4_outputs_e7/samples --num-uncond 10 --num-prefix 5 --temperatures 0.5 0.8 1.0 --top-k 50 --top-p 0.95 --render
Done. SAMPLES_DIR = /content/task4_outputs_e7/samples


## 6) Quantitative evaluation (test perplexity + validity rates)


In [17]:
import os, sys, subprocess

REPO_ROOT = os.environ["REPO_ROOT"]

CKPT = os.path.join(MODEL_DIR, "checkpoints", "final.pt")
CFG  = os.path.join(MODEL_DIR, "config.json")

# Use your actual Drive path for test.jsonl (or fall back to repo data/processed/test.jsonl)
TEST_PATH = "/content/drive/MyDrive/svg_task2_data/test.jsonl"

print("Using TEST_PATH =", TEST_PATH)
print("Using MODEL_DIR =", MODEL_DIR)
print("Using CKPT =", CKPT)
print("Using CFG =", CFG)
print("Using SAMPLES_DIR =", SAMPLES_DIR)
print("Will write EVAL_JSON =", EVAL_JSON)

cmd = [
    sys.executable, "-u", "-m", "scripts.task4.evaluate_generation",
    "--test-jsonl", TEST_PATH,
    "--spm-model", SPM,
    "--checkpoint", CKPT,
    "--config", CFG,
    "--samples-dir", SAMPLES_DIR,
    "--out-json", EVAL_JSON,
]

print("Running:", " ".join(cmd))

proc = subprocess.Popen(
    cmd,
    cwd=REPO_ROOT,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

for line in proc.stdout:
    print(line, end="")

ret = proc.wait()
if ret != 0:
    raise RuntimeError(f"Evaluation failed with exit code {ret}")

print("Done. EVAL_JSON =", EVAL_JSON)

Using TEST_PATH = /content/drive/MyDrive/svg_task2_data/test.jsonl
Using MODEL_DIR = /content/task4_outputs_e7/best_xl_sp
Using CKPT = /content/task4_outputs_e7/best_xl_sp/checkpoints/final.pt
Using CFG = /content/task4_outputs_e7/best_xl_sp/config.json
Using SAMPLES_DIR = /content/task4_outputs_e7/samples
Will write EVAL_JSON = /content/task4_outputs_e7/eval_metrics.json
Running: /usr/bin/python3 -u -m scripts.task4.evaluate_generation --test-jsonl /content/drive/MyDrive/svg_task2_data/test.jsonl --spm-model /content/drive/MyDrive/svg_task2_data/spm.model --checkpoint /content/task4_outputs_e7/best_xl_sp/checkpoints/final.pt --config /content/task4_outputs_e7/best_xl_sp/config.json --samples-dir /content/task4_outputs_e7/samples --out-json /content/task4_outputs_e7/eval_metrics.json
Wrote /content/task4_outputs_e7/eval_metrics.json
{
  "test_nll": 0.8271895855665207,
  "test_perplexity": 2.2868826125054245,
  "num_generated": 15,
  "xml_valid_rate": 0.6,
  "struct_valid_rate": 0.6,
  

In [10]:
!find /content -name "test.jsonl"

## 7) Rendered grid figure for the report


In [18]:
import os, sys, subprocess
REPO_ROOT = os.environ["REPO_ROOT"]

os.makedirs(FIG_DIR, exist_ok=True)

cmd = [
    sys.executable, "-u", "-m", "scripts.task4.figure_report",
    "--samples-dir", SAMPLES_DIR,
    "--out-dir", FIG_DIR
]

print("Running:", " ".join(cmd))
subprocess.check_call(cmd, cwd=REPO_ROOT)
print("Done. FIG_DIR =", FIG_DIR)

Running: /usr/bin/python3 -u -m scripts.task4.figure_report --samples-dir /content/task4_outputs_e7/samples --out-dir /content/task4_outputs_e7/figures_report
Done. FIG_DIR = /content/task4_outputs_e7/figures_report


In [19]:
import os, shutil
from pathlib import Path

# local -> drive
src_root = Path(TASK4_LOCAL)
dst_root = Path(TASK4_DRIVE)

# clean destination first
if dst_root.exists():
    shutil.rmtree(dst_root)
dst_root.mkdir(parents=True, exist_ok=True)

# copy only final artifacts
to_copy = [
    ("best_xl_sp", "best_xl_sp"),
    ("samples", "samples"),
    ("figures_report", "figures_report"),
    ("eval_metrics.json", "eval_metrics.json"),
]

for src_name, dst_name in to_copy:
    src = src_root / src_name
    dst = dst_root / dst_name
    if src.is_dir():
        shutil.copytree(src, dst, dirs_exist_ok=True)
        print("Copied folder:", src, "->", dst)
    elif src.is_file():
        shutil.copy2(src, dst)
        print("Copied file:", src, "->", dst)
    else:
        print("Missing (skip):", src)

print("Final outputs copied to Drive:", dst_root)

Copied folder: /content/task4_outputs_e7/best_xl_sp -> /content/drive/MyDrive/svg_task4_outputs_e7/best_xl_sp
Copied folder: /content/task4_outputs_e7/samples -> /content/drive/MyDrive/svg_task4_outputs_e7/samples
Copied folder: /content/task4_outputs_e7/figures_report -> /content/drive/MyDrive/svg_task4_outputs_e7/figures_report
Copied file: /content/task4_outputs_e7/eval_metrics.json -> /content/drive/MyDrive/svg_task4_outputs_e7/eval_metrics.json
Final outputs copied to Drive: /content/drive/MyDrive/svg_task4_outputs_e7
